# Lab 05: Cleanup

Tear down the **Bedrock Agent** (created in Lab 03). All other resources are
managed by CloudFormation — they will be deleted when the facilitator deletes the stack.

Only run this when the workshop is over.

## Load config

In [ ]:
# Config is written by the SageMaker lifecycle script from SSM at space startup.
# If this fails, re-launch the JupyterLab space to trigger the lifecycle script.
import json, pathlib, boto3

config = json.loads(pathlib.Path('/tmp/certagent_config.json').read_text())
globals().update(config)

REPO_DIR = pathlib.Path(REPO_DIR)
lm  = boto3.client('lambda',        region_name=AWS_REGION)
ddb = boto3.resource('dynamodb',    region_name=AWS_REGION)
sm  = boto3.client('secretsmanager',region_name=AWS_REGION)

PRIORITY_EMOJI = {'EXPIRED': '💀', 'CRITICAL': '🔴', 'HIGH': '🟠', 'MEDIUM': '🟡', 'LOW': '🟢'}

def invoke(fn, payload):
    r = lm.invoke(FunctionName=fn, InvocationType='RequestResponse',
                  Payload=json.dumps(payload))
    raw = json.loads(r['Payload'].read())
    if 'FunctionError' in r:
        raise RuntimeError(raw.get('errorMessage', raw))
    return raw.get('body', raw)

print(f'Region  : {AWS_REGION}')
print(f'Table   : {CERT_TABLE_NAME}')
print(f'Lambda  : {LAMBDA_SCAN}')
print('✅ Environment ready')

In [ ]:
iam_client = boto3.client('iam')
ACCOUNT_ID = boto3.client('sts').get_caller_identity()['Account']

## Delete Bedrock Agent

In [ ]:
if 'AGENT_ID' not in config:
    print('No agent found')
else:
    ba = boto3.client('bedrock-agent', region_name=AWS_REGION)
    if 'AGENT_ALIAS_ID' in config:
        try: ba.delete_agent_alias(agentId=config['AGENT_ID'], agentAliasId=config['AGENT_ALIAS_ID'])
        except: pass
        print(f'Alias deleted')
    try: ba.delete_agent(agentId=config['AGENT_ID'], skipResourceInUseCheck=True)
    except Exception as e: print(e)
    print(f'Agent deleted: {config["AGENT_ID"]}')

## Delete agent IAM role

In [ ]:
ROLE_NAME = f'{WORKSHOP_PREFIX}-bedrock-agent-role'
try:
    iam_client.delete_role_policy(RoleName=ROLE_NAME, PolicyName='AgentPolicy')
    iam_client.delete_role(RoleName=ROLE_NAME)
    print(f'Deleted role: {ROLE_NAME}')
except iam_client.exceptions.NoSuchEntityException:
    print(f'Role not found: {ROLE_NAME}')

## Delete Secrets Manager secrets created during labs

In [ ]:
paginator = sm.get_paginator('list_secrets')
to_delete = []
for page in paginator.paginate(Filters=[{'Key':'name','Values':[f'/{WORKSHOP_PREFIX}/certs']}]):
    for s in page['SecretList']: to_delete.append(s['Name'])
print(f'Secrets to delete: {len(to_delete)}')
for name in to_delete:
    sm.delete_secret(SecretId=name, ForceDeleteWithoutRecovery=True)
    print(f'  Deleted: {name}')
print('Done')

## Workshop Complete!

The remaining infrastructure (Lambda, DynamoDB, SNS, EventBridge, SageMaker domain)
is managed by CloudFormation. Delete the **workshop-master** stack to remove everything.

Key takeaways:
- Bedrock Agents can orchestrate multi-step workflows via Lambda action groups
- OpenAPI schemas define the tool interface — no agent code needed
- Mock-first development lets you build and test without external dependencies
- EventBridge + SNS provides automated proactive monitoring